# Exploração da API SIDRA do IBGE

Objetivo: entender a estrutura de resposta do SIDRA, começando pela
tabela 5932 (PIB trimestral).

**Referência:** https://servicodados.ibge.gov.br/api/docs/agregados?versao=3

In [ ]:
import requests
import json

URL_METADADOS = "https://servicodados.ibge.gov.br/api/v3/agregados/5932/metadados"

resposta = requests.get(URL_METADADOS, timeout=15)
print(f"Status: {resposta.status_code}")
print(f"Content-Type: {resposta.headers.get('Content-Type')}")

metadados = resposta.json()
print(f"\nTipo: {type(metadados)}")

In [ ]:
# metadados É um dicionário, não uma lista
tabela = metadados  # renomeando por clareza

print(f"Nome da tabela: {tabela.get('nome', 'N/A')}")
print(f"Periodicidade: {tabela.get('periodicidade', {}).get('texto', 'N/A')}")
print()

print("Variáveis disponíveis:")
for v in tabela.get("variaveis", []):
    print(f"  {v['id']} — {v['nome']} ({v.get('unidade', 'N/A')})")

In [ ]:
print("Classificações disponíveis:")
for c in tabela.get("classificacoes", []):
    print(f"\n  ID {c['id']} — {c['nome']}")
    for cat in c.get("categorias", []):  # sem [:5]
        print(f"    {cat['id']} — {cat['nome']}")

In [ ]:
# Visão geral: quantas categorias cada classificação tem
print("Classificações (visão geral):\n")
for c in tabela.get("classificacoes", []):
    n_cats = len(c.get("categorias", []))
    print(f"  ID {c['id']} — {c['nome']}  ({n_cats} categorias)")

In [ ]:
print("Classificações (só nomes):")
for c in tabela.get("classificacoes", []):
    print(f"  ID {c['id']} — {c['nome']} ({len(c.get('categorias', []))} categorias)")

In [ ]:
# Detalhar as categorias da classificação de "componentes da demanda"
# (substitua o ID pelo que aparecer na célula acima)
CLASSIFICACAO_DEMANDA = 11255  # ajuste conforme o que aparecer

for c in tabela.get("classificacoes", []):
    if c["id"] == CLASSIFICACAO_DEMANDA:
        for cat in c.get("categorias", []):
            print(f"  {cat['id']} — {cat['nome']}")

In [ ]:
# Ver as chaves do dicionário de nível superior
print("Chaves de nível superior:")
for chave in metadados.keys():
    valor = metadados[chave]
    tipo = type(valor).__name__
    print(f"  {chave!r}: ({tipo})")

In [ ]:
# Mapa do dicionário: quais chaves existem e de que tipo
print("Chaves de nível superior:")
for chave in metadados.keys():
    valor = metadados[chave]
    tipo = type(valor).__name__
    print(f"  {chave!r}: ({tipo})")

In [ ]:
metadados

In [ ]:
# Testando uma chamada real: PIB a preços de mercado (90707), taxa trimestral (6561)
URL = "https://servicodados.ibge.gov.br/api/v3/agregados/5932/periodos/-4/variaveis/6561?localidades=N1[all]&classificacao=11255[90707]"

r = requests.get(URL, timeout=15)
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
print()

# Ver o JSON formatado (primeiras linhas)
import json
print(json.dumps(r.json(), indent=2, ensure_ascii=False)[:2000])

In [ ]:
print(json.dumps(tabela, indent=2, ensure_ascii=False))

In [ ]:
# Testar múltiplas categorias numa única chamada
CATEGORIAS = "90707,93404,93405,93406,93407,93408,90687,90691,90696,90706"

URL = (
    "https://servicodados.ibge.gov.br/api/v3/agregados/5932"
    "/periodos/-4"
    "/variaveis/6561"
    f"?localidades=N1[all]"
    f"&classificacao=11255[{CATEGORIAS}]"
)

print(f"URL: {URL}\n")

r = requests.get(URL, timeout=15)
print(f"Status: {r.status_code}")

dados = r.json()
print(f"Tipo: {type(dados)}")
print(f"Quantidade de variáveis retornadas: {len(dados)}")

In [ ]:
# Ver a estrutura geral
resultado = dados[0]
print(f"Variável: {resultado['variavel']}")
print(f"Unidade: {resultado['unidade']}")
print(f"Quantos resultados: {len(resultado['resultados'])}")

print("\n--- Estrutura dos resultados ---")
for res in resultado["resultados"]:
    print(f"\nClassificação: {res['classificacoes'][0]['nome']}")
    print(f"Categorias neste resultado:")
    for cat_id, cat_nome in res["classificacoes"][0]["categoria"].items():
        print(f"  {cat_id} → {cat_nome}")
    print(f"Séries: {len(res['series'])} localidades")

In [ ]:
# Ver os valores
for res in resultado["resultados"]:
    cats = res["classificacoes"][0]["categoria"]
    cat_id = list(cats.keys())[0]
    cat_nome = cats[cat_id]

    for serie in res["series"]:
        valores = serie["serie"]
        print(f"{cat_id} ({cat_nome}):")
        for periodo, valor in valores.items():
            print(f"  {periodo}: {valor}")